In [77]:
import pandas as pd

In [78]:
dist = pd.read_csv('data/school_districts.csv')

In [79]:
len(dist)

54

In [80]:
df = pd.read_csv('data/MVS_data_WA.csv')

In [81]:
import re
from rapidfuzz import process, fuzz

# strip generic suffixes/district numbers so matching is driven by the distinctive part of the name
# (otherwise "School District"/"Public Schools" being shared tokens inflates scores for unrelated districts)
# lookahead (not \b) after the optional "." since punctuation followed by whitespace has no word boundary
GENERIC_SUFFIXES = re.compile(
    r'\b(School District|Public Schools|Community Schools|Schools|Historical|Hist\.?|Elementary)(?=\s|$)',
    re.IGNORECASE,
)
DISTRICT_NUMBER = re.compile(r'(No\.?\s*)?#?\s*\d+\s*$', re.IGNORECASE)

def normalize(name):
    name = DISTRICT_NUMBER.sub('', name)
    name = GENERIC_SUFFIXES.sub('', name)
    return re.sub(r'\s+', ' ', name).strip(' .')

lea_names = df['LEA'].dropna().unique()
dist_names = dist['School District'].dropna().unique()
normalized_dist = {normalize(d): d for d in dist_names}

lea_to_dist = {}
for lea in lea_names:
    match = process.extractOne(
        normalize(lea), normalized_dist.keys(), scorer=fuzz.ratio, score_cutoff=95
    )
    lea_to_dist[lea] = normalized_dist[match[0]] if match else None

matched_pairs = {lea: d for lea, d in lea_to_dist.items() if d is not None}
unmatched = [lea for lea, d in lea_to_dist.items() if d is None]
print(f"{len(matched_pairs)} of {len(lea_names)} LEA names matched a district:")
matched_pairs


54 of 343 LEA names matched a district:


{'Clover Park School District': 'Clover Park School District',
 'Highline School District': 'Highline Public Schools',
 'Marysville School District': 'Marysville School District',
 'North Mason School District': 'North Mason School District',
 'Seattle School District No. 1': 'Seattle Public Schools',
 'Sumner-Bonney Lake School District': 'Sumner-Bonney Lake School District',
 'Tacoma School District': 'Tacoma Public Schools',
 'Yelm School District': 'Yelm Community Schools',
 'Enumclaw School District': 'Enumclaw School District',
 'Arlington School District': 'Arlington Public Schools',
 'Auburn School District': 'Auburn School District',
 'Bainbridge Island School District': 'Bainbridge Island School District',
 'Bellevue School District': 'Bellevue School District',
 'Bethel School District': 'Bethel School District',
 'Bremerton School District': 'Bremerton School District',
 'Carbonado School District': 'Carbonado Historical School District',
 'Central Kitsap School District': 

In [82]:
df_matched = df[df['LEA'].map(lea_to_dist).notna()].copy()
df_matched['School District'] = df_matched['LEA'].map(lea_to_dist)
df_matched


,School Year,State,NCES LEA ID,LEA,School,NCES SCH ID,Data Group,Data Description,Value,Denominator,Numerator,Population,Subgroup,Characteristics,Age/Grade,Academic Subject,Program Type,Outcome,School District
35,2023-2024,WASHINGTON,5301410.0,Clover Park School District,NaN,NaN,818,Young homeless children served by McKinney-Vento,3,NaN,NaN,Homeless Students,All Students in LEA,NaN,Age birth through 5 (not kindergarten),NaN,NaN,NaN,Clover Park School District
41,2023-2024,WASHINGTON,5303540.0,Highline School District,NaN,NaN,818,Young homeless children served by McKinney-Vento,50,NaN,NaN,Homeless Students,All Students in LEA,NaN,Age birth through 5 (not kindergarten),NaN,NaN,NaN,Highline Public Schools
44,2023-2024,WASHINGTON,5304860.0,Marysville School District,NaN,NaN,818,Young homeless children served by McKinney-Vento,12,NaN,NaN,Homeless Students,All Students in LEA,NaN,Age birth through 5 (not kindergarten),NaN,NaN,NaN,Marysville School District
49,2023-2024,WASHINGTON,5305790.0,North Mason School District,NaN,NaN,818,Young homeless children served by McKinney-Vento,55,NaN,NaN,Homeless Students,All Students in LEA,NaN,Age birth through 5 (not kindergarten),NaN,NaN,NaN,North Mason School District
54,2023-2024,WASHINGTON,5307710.0,Seattle School District No. 1,NaN,NaN,818,Young homeless children served by McKinney-Vento,19,NaN,NaN,Homeless Students,All Students in LEA,NaN,Age birth through 5 (not kindergarten),NaN,NaN,NaN,Seattle Public Schools
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2824,2023-2024,WASHINGTON,5300390.0,Bellevue School District,NaN,NaN,655,Homeless Students Enrolled,70,NaN,NaN,Homeless Students,Black or African American,NaN,Pre-K through 12th grade,NaN,NaN,NaN,Bellevue School District
2826,2023-2024,WASHINGTON,5308040.0,Snoqualmie Valley School District,NaN,NaN,655,Homeless Students Enrolled,4,NaN,NaN,Homeless Students,Black or African American,NaN,Pre-K through 12th grade,NaN,NaN,NaN,Snoqualmie Valley School District
2827,2023-2024,WASHINGTON,5310140.0,Yelm School District,NaN,NaN,655,Homeless Students Enrolled,4,NaN,NaN,Homeless Students,Black or African American,NaN,Pre-K through 12th grade,NaN,NaN,NaN,Yelm Community Schools
2828,2023-2024,WASHINGTON,5304230.0,Lake Washington School District,NaN,NaN,655,Homeless Students Enrolled,83,NaN,NaN,Homeless Students,Black or African American,NaN,Pre-K through 12th grade,NaN,NaN,NaN,Lake Washington School District


In [85]:
df_matched.loc[df['Characteristics'] == 'Doubled-up'][['LEA','Value']].sort_values(by='LEA').rename(columns={'Value': 'Doubled-up students'}).to_csv('output/intermediate/doubled_up_students.csv')

In [92]:
df_matched.loc[df['Characteristics'] == 'Doubled-up']['Value'].str.replace(',','').apply(pd.to_numeric).sum()

np.int64(17712)

In [94]:
xwalk = pd.read_csv("output/intermediate/puma_geog_lookup_cnty.csv")
h = (
    pd.read_csv('output/intermediate/psam_h53.csv')
    .query('PUMA.isin(@xwalk["PUMA"]) & TYPEHUGQ == 1')
)

In [96]:
p = (
pd.read_csv('output/intermediate/psam_p53.csv')
.query('SERIALNO.isin(@h["SERIALNO"])')
)

In [108]:
age_mask = (p['AGEP'] >= 5) & (p['AGEP'] < 18)
children = p.loc[age_mask].groupby('SERIALNO').size()
hh_children = h.merge(children.rename('num_children'), on='SERIALNO', how='inner')

In [109]:
hh_children

,RT,SERIALNO,DIVISION,PUMA,REGION,STATE,ADJHSG,ADJINC,WGTP,NP,...,WGTP72,WGTP73,WGTP74,WGTP75,WGTP76,WGTP77,WGTP78,WGTP79,WGTP80,num_children
0,H,2024HU0000041,9,23501,4,53,1000000,1015250,89,6,...,155,24,96,164,26,24,79,96,101,2
1,H,2024HU0000171,9,25303,4,53,1000000,1015250,56,4,...,18,78,16,15,55,59,59,49,19,2
2,H,2024HU0001183,9,23311,4,53,1000000,1015250,303,5,...,384,252,259,92,310,105,100,567,335,2
3,H,2024HU0001211,9,26105,4,53,1000000,1015250,68,3,...,83,71,104,125,21,18,96,73,69,1
4,H,2024HU0001503,9,23501,4,53,1000000,1015250,9,8,...,7,9,17,8,11,13,4,3,10,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3597,H,2024HU1462431,9,25308,4,53,1000000,1015250,68,4,...,131,72,64,62,23,68,121,76,69,2
3598,H,2024HU1462604,9,23306,4,53,1000000,1015250,150,5,...,223,47,169,226,187,136,128,170,174,2
3599,H,2024HU1463270,9,26103,4,53,1000000,1015250,131,4,...,271,37,41,268,123,37,241,234,118,1
3600,H,2024HU1463528,9,26101,4,53,1000000,1015250,51,4,...,75,52,16,85,15,56,47,49,52,1


In [112]:
# weighted average number of school-age (5-17) kids, among households that have at least one

avg_children_per_hh = (hh_children['num_children'] * hh_children['WGTP']).sum() / hh_children['WGTP'].sum()
avg_children_per_hh

np.float64(1.6371076454058089)

In [113]:
total_doubled_up_students = df_matched.loc[df['Characteristics'] == 'Doubled-up']['Value'].str.replace(',', '').apply(pd.to_numeric).sum()

doubled_up_households = total_doubled_up_students / avg_children_per_hh
doubled_up_households

np.float64(10819.080864783036)